# 🥾 Side Trails · Step 3 — Map a Job onto a GPU
**A Billion Small Steps · Hardware for AI · Step 3 · Book sections App. C (from 3.4) · Foothold lessons 3.C**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

| Part | What you do | Book |
|---|---|---|
| A | Measure this GPU's best FP32/FP16 speed | 3.1 (Core Lab 6 Part A again) |
| B | Measure its best bandwidth | 3.4 (Core Lab 7 Part B again) |
| C | Your plan vs reality | App. C |

⏱️ About 1 hour. 📘 Textbook: Hwu, Kirk & El Hajj Ch 4–5 · H&P §4.4. Course site: https://aishugithub.github.io/course_GPU/

🥾 **Side Trail: leaves from 3.4 → rejoin at Step 4.** Parts A and B re-measure the two numbers the plan needs.

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

---
## Part A · This GPU's best speed  *(Section 3.1)*

In [ ]:
# the T4's published design: 40 SMs, each with 64 FP32 cores and 8 Tensor Cores, boost clock 1.59 GHz
t4_sms, clock_ghz = 40, 1.59
# FP32: SMs × 64 cores × 2 FLOPs per fused multiply-add × clocks per second
fp32_peak = t4_sms * 64 * 2 * clock_ghz * 1e9
# FP16 Tensor Cores: SMs × 8 Tensor Cores × 64 multiply-adds each × 2 FLOPs × clocks per second
fp16_peak = t4_sms * 8 * 64 * 2 * clock_ghz * 1e9
print(f"T4 peak FP32 (CUDA cores)  : {fp32_peak / 1e12:5.1f} TFLOPS")   # ÷ 1e12 gives TFLOPS
print(f"T4 peak FP16 (Tensor Cores): {fp16_peak / 1e12:5.1f} TFLOPS")   # the Tensor Cores' peak
# the GPU in THIS session: how many SMs does it have?
sms = torch.cuda.get_device_properties(0).multi_processor_count
print(f"This GPU has {sms} SMs")   # show it

In [ ]:
N = 4096   # the matrices are 4096 × 4096
measured = {}   # number format → measured TFLOPS
# dtype is the number format: 32-bit floats, then 16-bit floats
for dtype in [torch.float32, torch.float16]:
    # two random 4096 × 4096 matrices on the GPU, in this format
    a = torch.rand(N, N, device=device, dtype=dtype); b = torch.rand(N, N, device=device, dtype=dtype)
    # average time of one multiply; speed = 2N³ FLOPs ÷ time, ÷ 1e12 gives TFLOPS
    t = timeit(lambda: a @ b, repeats=10)
    measured[dtype] = 2 * N**3 / t / 1e12   # speed = 2N³ FLOPs ÷ time, in TFLOPS
    print(f"{str(dtype):15s}: {t*1e3:7.2f} ms  →  {measured[dtype]:5.1f} TFLOPS measured")   # show the time and the speed
print(f"FP16 is {measured[torch.float16] / measured[torch.float32]:.1f}× faster than FP32 on this GPU")   # compare the two formats

---
## Part B · This GPU's best bandwidth  *(Section 3.4)*

In [ ]:
# 13.4 crore floats = 512 MB; .clone() copies it inside GPU memory: the best case for bandwidth
big = torch.rand(1 << 27, device=device)
t = timeit(lambda: big.clone(), repeats=5)   # time one copy
# bandwidth = bytes read + bytes written ÷ time
best_bw = 2 * big.numel() * 4 / t
print(f"Copy 512 MB inside the GPU: {t*1e3:.1f} ms → {best_bw / 1e9:.0f} GB/s  (a T4's published peak: 320 GB/s)")   # show the bandwidth

## Part C · Your Plan vs Reality  *(Section 3.C)*

The capstone planner predicts lower bounds from FLOPs and bytes, using **this GPU's own best speeds** measured in Parts A and B. Then we measure.

In [ ]:
# plan with this GPU's own best speeds, measured in Parts A and B
PEAK, BW = measured[torch.float32] * 1e12, best_bw
N = 4096   # the matrices are 4096 × 4096
# the layer's 4096 × 4096 weight matrix
W = torch.rand(N, N, device=device)
# the table heading (:16s and :>10s pad to a fixed width so the columns line up)
print(f"{'job':16s} {'plan (ms)':>10s} {'limit':>8s} {'measured':>10s} {'achieved':>14s}")
for B in [1, 64, 1024]:   # three batch sizes
    # a batch of B inputs
    X = torch.rand(B, N, device=device)
    # the layer: FLOPs = 2·B·N·N; bytes = read X, read W, write the output (4 bytes per number)
    flops, nbytes = 2 * B * N * N, (B * N + N * N + B * N) * 4
    # the plan is the bigger of the compute time and the memory time; then measure the reality
    plan_ms = max(flops / PEAK, nbytes / BW) * 1e3
    t = timeit(lambda: X @ W, repeats=10)   # the reality: time the layer
    print(f"matmul B={B:<6d} {plan_ms:10.3f} {'compute' if flops/PEAK > nbytes/BW else 'memory':>8s} {t*1e3:10.3f}   {flops / t / 1e12:6.2f} TFLOPS")   # plan, limit, measured time, achieved speed
    # ReLU on the output: almost no maths, it just reads and writes B·N numbers
    Y = X @ W
    nbytes = 2 * B * N * 4   # ReLU reads and writes B × N numbers of 4 bytes
    t = timeit(lambda: torch.relu(Y), repeats=10)   # time it
    print(f"ReLU   B={B:<6d} {nbytes/BW*1e3:10.3f} {'memory':>8s} {t*1e3:10.3f}   {nbytes / t / 1e9:6.1f} GB/s")   # plan, limit, measured time, achieved bandwidth

### ✏️ YOUR TURN (C) — record your results
| Job | Planned limit | Measured time | Achieved |
|---|---|---|---|
| matmul B=1 | | | |
| matmul B=1024 | | | |
| ReLU B=1024 | | | |

1. Which jobs came closest to their plan? Which were furthest away, and why might tiny jobs miss by a lot? (Hint: launch overhead, Section 2.5.)
2. Why does the B = 1024 matmul achieve far more TFLOPS than B = 1?

*Your answers:*